# 🚀 Huấn Luyện YOLOv8-Small Cho PCBDetect Trên Google Colab (GPU T4)

Notebook này được thiết kế để huấn luyện mô hình **YOLOv8s (11.2M tham số)** cho dự án **PCBDetect** trên GPU NVIDIA T4 miễn phí của Google Colab.

### ⚡ Hướng dẫn nhanh:
1. Chọn menu: **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Bấm **Save**.
2. Kéo thả file `yolo_dataset.zip` (34MB từ máy tính của bạn) vào tab **Files** (thư mục bên trái của Colab).
3. Chạy lần lượt các ô code bên dưới (bấm Shift + Enter)!

In [ ]:
# 1. Kiểm tra phần cứng GPU T4
!nvidia-smi

In [ ]:
# 2. Cài đặt thư viện Ultralytics YOLOv8
!pip install -q ultralytics

In [ ]:
# 3. Giải nén tập dữ liệu DeepPCB (yolo_dataset.zip)
!unzip -q yolo_dataset.zip -d dataset/
!ls -lh dataset/yolo

In [ ]:
# 4. Tạo file cấu hình data.yaml
import yaml

data_yaml = {
    'path': '/content/dataset/yolo',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'open',
        1: 'short',
        2: 'mousebite',
        3: 'spur',
        4: 'copper',
        5: 'pin_hole'
    },
    'nc': 6
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Đã tạo file /content/data.yaml thành công!")

In [ ]:
# 5. Huấn luyện mô hình YOLOv8s với GPU T4 (50 Epochs)
# Trên GPU T4, 50 epochs chỉ mất khoảng 8 - 10 phút!
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,          # Sử dụng GPU 0 (NVIDIA T4)
    box=7.5,           # Tăng trọng số box_loss cho vật thể siêu nhỏ
    cls=0.5,
    dfl=1.5,
    degrees=90.0,      # Bất biến quay bo mạch 90/180 độ
    fliplr=0.5,
    flipud=0.5,
    mosaic=1.0,
    close_mosaic=10,
    project='pcb_runs',
    name='yolov8s_pcb',
    patience=15,       # Tự động dừng sớm nếu sau 15 epoch điểm không tăng thêm
    exist_ok=True,
    verbose=True
)

In [ ]:
# 6. Đo kiểm chất lượng độc lập trên tập Test (234 bo mạch)
test_metrics = model.val(data='/content/data.yaml', split='test', device=0)

print("\n================ KẾT QUẢ ĐO KIỂM TẬP TEST ================")
print(f"🎯 mAP@0.5      : {test_metrics.box.map50 * 100:.2f}%")
print(f"🎯 mAP@0.5:0.95 : {test_metrics.box.map * 100:.2f}%")
print(f"🎯 Precision    : {test_metrics.box.mp * 100:.2f}%")
print(f"🎯 Recall       : {test_metrics.box.mr * 100:.2f}%")
f1 = 2 * test_metrics.box.mp * test_metrics.box.mr / (test_metrics.box.mp + test_metrics.box.mr + 1e-9)
print(f"🎯 F1-Score     : {f1 * 100:.2f}%")

In [ ]:
# 7. Tự động tải file trọng số best.pt về máy tính của bạn
from google.colab import files

best_weight_path = 'pcb_runs/yolov8s_pcb/weights/best.pt'
print(f"Đang chuẩn bị tải {best_weight_path} về máy...")
files.download(best_weight_path)